<a href="https://colab.research.google.com/github/snumryk/TRPA1-ML-benchmark/blob/main/scripts/D_MPNN_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chemprop rdkit -q
import chemprop, torch
print(f"Chemprop {chemprop.__version__}, torch {torch.__version__}, CUDA {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.5/149.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 76.1 MB/s eta 0:00:00
Chemprop 2.2.4, torch 2.11.0+cu128, CUDA True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
D-MPNN GroupKFold CV in Colab (local torch is broken on Windows).
Data loaded from Drive. Same 5-fold scaffold split.
"""
import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, matthews_corrcoef
from scipy.stats import spearmanr
from chemprop.data import MoleculeDatapoint, MoleculeDataset, build_dataloader
from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN
from chemprop.models import MPNN
import warnings
warnings.filterwarnings('ignore')

SEED = 42
THRESHOLD = 7.0
DATA_PATH = '/content/drive/MyDrive/trpa1_project'

df = pd.read_csv(f'{DATA_PATH}/trpa1_antagonists.csv')
y = df['pchembl_median'].values
scaffolds = df['scaffold'].values
smiles = df['std_smiles'].values
print(f"Compounds: {len(df)}, scaffolds: {len(np.unique(scaffolds))}")

RDKIT_DESCS = [
    'MolWt', 'MolLogP', 'MolMR', 'TPSA',
    'NumHAcceptors', 'NumHDonors', 'NumRotatableBonds',
    'NumAromaticRings', 'RingCount', 'FractionCSP3',
    'HeavyAtomCount', 'NumAliphaticRings', 'NumSaturatedRings',
    'NumHeteroatoms', 'LabuteASA',
]

def compute_rdkit(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(len(RDKIT_DESCS), dtype=np.float32)
    return np.array([float(getattr(Descriptors, n)(mol)) for n in RDKIT_DESCS], dtype=np.float32)

def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    rho = spearmanr(y_true, y_pred).correlation
    y_cls = (y_true >= THRESHOLD).astype(int)
    if len(np.unique(y_cls)) < 2:
        return rmse, r2, rho, np.nan, np.nan
    auc = roc_auc_score(y_cls, y_pred)
    mcc = matthews_corrcoef(y_cls, (y_pred >= THRESHOLD).astype(int))
    return rmse, r2, rho, auc, mcc

def ms(vals):
    vals = np.array(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    return np.mean(vals), np.std(vals)

gkf = GroupKFold(n_splits=5)

def run_dmpnn_cv(use_rdkit):
    fm = {'rmse': [], 'r2': [], 'rho': [], 'auc': [], 'mcc': []}
    for fold, (tr, te) in enumerate(gkf.split(smiles, y, groups=scaffolds)):
        rng = np.random.default_rng(SEED + fold)
        tr_sh = rng.permutation(tr)
        n_val = int(0.15 * len(tr_sh))
        val_idx, fit_idx = tr_sh[:n_val], tr_sh[n_val:]

        def make_points(indices):
            pts = []
            for i in indices:
                mol = Chem.MolFromSmiles(smiles[i])
                if mol is None:
                    continue
                if use_rdkit:
                    pts.append(MoleculeDatapoint(mol=mol, y=np.array([y[i]]), x_d=compute_rdkit(smiles[i])))
                else:
                    pts.append(MoleculeDatapoint(mol=mol, y=np.array([y[i]])))
            return pts

        fit_loader = build_dataloader(MoleculeDataset(make_points(fit_idx)), batch_size=64, shuffle=True, seed=SEED)
        val_loader = build_dataloader(MoleculeDataset(make_points(val_idx)), batch_size=64, shuffle=False)
        test_loader = build_dataloader(MoleculeDataset(make_points(te)), batch_size=64, shuffle=False)

        mp = BondMessagePassing(d_h=300, depth=3)
        ffn = RegressionFFN(input_dim=300 + len(RDKIT_DESCS)) if use_rdkit else RegressionFFN()
        model = MPNN(message_passing=mp, agg=MeanAggregation(), predictor=ffn,
                     warmup_epochs=2, init_lr=1e-4, max_lr=1e-3, final_lr=1e-4)

        early = EarlyStopping(monitor='val_loss', patience=10, mode='min')
        trainer = L.Trainer(max_epochs=100, accelerator='auto',
                            enable_progress_bar=False, enable_model_summary=False,
                            logger=False, callbacks=[early])
        trainer.fit(model, fit_loader, val_loader)

        preds = torch.cat(trainer.predict(model, test_loader)).numpy().ravel()
        actuals = np.array([y[i] for i in te if Chem.MolFromSmiles(smiles[i]) is not None])
        for k, v in zip(['rmse','r2','rho','auc','mcc'], metrics(actuals, preds)):
            fm[k].append(v)
        print(f"    fold {fold+1}/5: R2={fm['r2'][-1]:.3f}")
    return {k: ms(fm[k]) for k in fm}

print("\nD-MPNN (graph only)...")
r_graph = run_dmpnn_cv(False)
print(f"  R2={r_graph['r2'][0]:.3f}±{r_graph['r2'][1]:.3f}, AUC={r_graph['auc'][0]:.3f}±{r_graph['auc'][1]:.3f}")

print("\nD-MPNN+RDKit...")
r_rdkit = run_dmpnn_cv(True)
print(f"  R2={r_rdkit['r2'][0]:.3f}±{r_rdkit['r2'][1]:.3f}, AUC={r_rdkit['auc'][0]:.3f}±{r_rdkit['auc'][1]:.3f}")

# ── Final combined table ──────────────────────────────────────
print("\n" + "="*95)
print("COMPLETE CV TABLE — all models, 5-fold GroupKFold by scaffold, mean ± std")
print("="*95)
print(f"{'Model':<22} {'R2':>14} {'RMSE':>14} {'Spearman':>14} {'AUC':>14}")
print("-"*95)

full = {
    'RF (Morgan)':       {'r2': (0.547, 0.112), 'rmse': (0.618, 0.050), 'rho': (0.000, 0.0), 'auc': (0.856, 0.060)},
    'XGB CB-CLS':        {'r2': (0.539, 0.084), 'rmse': (0.633, 0.057), 'rho': (0.725, 0.064), 'auc': (0.866, 0.043)},
    'XGB (Morgan)':      {'r2': (0.539, 0.106), 'rmse': (0.625, 0.046), 'rho': (0.000, 0.0), 'auc': (0.860, 0.052)},
    'XGB MF-Mean':       {'r2': (0.529, 0.085), 'rmse': (0.639, 0.042), 'rho': (0.703, 0.066), 'auc': (0.857, 0.045)},
    'RF (RDKit-15)':     {'r2': (0.478, 0.163), 'rmse': (0.664, 0.053), 'rho': (0.695, 0.089), 'auc': (0.848, 0.051)},
    'D-MPNN':            r_graph,
    'D-MPNN+RDKit':      r_rdkit,
}

for name in sorted(full, key=lambda k: full[k]['r2'][0], reverse=True):
    r = full[name]
    print(f"{name:<22} {r['r2'][0]:>7.3f}±{r['r2'][1]:.3f} "
          f"{r['rmse'][0]:>7.3f}±{r['rmse'][1]:.3f} "
          f"{r['rho'][0]:>7.3f}±{r['rho'][1]:.3f} "
          f"{r['auc'][0]:>7.3f}±{r['auc'][1]:.3f}")
print("-"*95)

import json
with open(f'{DATA_PATH}/dmpnn_cv_results.json', 'w') as f:
    json.dump({'D-MPNN': r_graph, 'D-MPNN+RDKit': r_rdkit}, f, indent=2)
print("\nSaved dmpnn_cv_results.json")

Compounds: 1645, scaffolds: 544

D-MPNN (graph only)...


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

    fold 1/5: R2=0.254


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

    fold 2/5: R2=0.532


...

    fold 3/5: R2=0.410


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

    fold 4/5: R2=0.517


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

    fold 5/5: R2=0.588
  R2=0.491±0.092, AUC=0.843±0.049

COMPLETE CV TABLE — all models, 5-fold GroupKFold by scaffold, mean ± std
Model                              R2           RMSE       Spearman            AUC
-----------------------------------------------------------------------------------------------
RF (Morgan)              0.547±0.112   0.618±0.050   0.000±0.000   0.856±0.060
XGB CB-CLS               0.539±0.084   0.633±0.057   0.725±0.064   0.866±0.043
XGB (Morgan)             0.539±0.106   0.625±0.046   0.000±0.000   0.860±0.052
XGB MF-Mean              0.529±0.085   0.639±0.042   0.703±0.066   0.857±0.045
D-MPNN+RDKit             0.491±0.092   0.663±0.040   0.677±0.086   0.843±0.049
RF (RDKit-15)            0.478±0.163   0.664±0.053   0.695±0.089   0.848±0.051
D-MPNN                   0.410±0.107   0.714±0.031   0.597±0.089   0.794±0.050
-----------------------------------------------------------------------------------------------

Saved dmpnn_cv_results.json
